# Phase 5 v2 — Fine-Tuning



In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Avoids deadlock with dataloader workers

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/ImmunoPath"
DATA_DIR = f"{PROJECT_DIR}/data"
TRAINING_DIR = f"{DATA_DIR}/training_v2"
MODEL_DIR = f"{PROJECT_DIR}/models/immunopath-v2"
RESULTS_DIR = f"{PROJECT_DIR}/results/phase5_v2"

# Local SSD paths — CRITICAL for speed (GDrive I/O is 10-50x slower)
LOCAL_CKPT_DIR = "/content/immunopath_ckpts_v2"
LOCAL_LOG_DIR = "/content/immunopath_logs_v2"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
os.makedirs(LOCAL_LOG_DIR, exist_ok=True)

# Install exact versions for reproducibility
import subprocess
subprocess.run([
    "pip", "install", "-q", "--no-cache-dir",
    "transformers>=4.50.0",
    "accelerate>=0.34.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.44.0",
    "trl>=0.12.0",
    "datasets",
    "tqdm",
], check=True)

# Flash Attention 2 — critical for A100 performance
try:
    subprocess.run(["pip", "install", "-q", "flash-attn", "--no-build-isolation"], check=True)
    FLASH_ATTN_AVAILABLE = True
    print("Flash Attention 2 installed ✓")
except Exception:
    FLASH_ATTN_AVAILABLE = False
    print("Flash Attention not available — using eager attention")

from huggingface_hub import login
from google.colab import userdata
try:
    login(token=userdata.get("HF_TOKEN"))
    print("HuggingFace login ✓")
except Exception:
    print("Set HF_TOKEN in Colab Secrets")

import torch
import gc

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

    # A100-specific: Enable TF32 (free ~2x speedup for matmul)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

    # Verify A100
    if "A100" in gpu_name:
        print("  ✓ A100 detected — TF32 + Flash Attention will be used")
    elif "L4" in gpu_name:
        print("    L4 detected — may OOM with batch_size=4. Consider using A100.")
else:
    print("  No GPU detected — training will fail!")

import transformers
print(f"transformers: {transformers.__version__}")
import peft; print(f"peft: {peft.__version__}")
import trl; print(f"trl: {trl.__version__}")
print(f"\nLocal SSD paths:")
print(f"  Checkpoints: {LOCAL_CKPT_DIR}")
print(f"  Logs:        {LOCAL_LOG_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Flash Attention 2 installed ✓
HuggingFace login ✓
GPU: NVIDIA A100-SXM4-80GB (85.1 GB)
  ✓ A100 detected — TF32 + Flash Attention will be used


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


transformers: 5.0.0
peft: 0.18.1
trl: 0.28.0

Local SSD paths:
  Checkpoints: /content/immunopath_ckpts_v2
  Logs:        /content/immunopath_logs_v2


## Configuration

In [3]:
from dataclasses import dataclass, field
from typing import Tuple

@dataclass
class Config:
    # Model
    model_id: str = "google/medgemma-1.5-4b-it"

    # LoRA — following Google's official notebook EXACTLY
    lora_r: int = 16
    lora_alpha: int = 16               # Google uses alpha=16
    lora_dropout: float = 0.05
    use_dora: bool = False             # Google doesn't use DoRA — we follow
    target_modules: str = "all-linear" # FIX #6: full linear coverage
    modules_to_save: list = field(default_factory=lambda: ["lm_head", "embed_tokens"])

    # Training — Google's exact settings
    num_epochs: int = 1
    batch_size: int = 4
    grad_accum: int = 4
    lr: float = 2e-4                  # Google's exact LR
    warmup_ratio: float = 0.03        # QLoRA paper
    weight_decay: float = 0.01
    max_grad_norm: float = 0.3
    lr_scheduler_type: str = "linear" # Google's setting

    # Data
    max_patches: int = 4              # Keep 4 for memory
    max_length: int = 2048            # Needed for multi-image (4×256 img tokens + text)

    # Paths
    train_path: str = f"{TRAINING_DIR}/train.jsonl"
    val_path: str = f"{TRAINING_DIR}/val.jsonl"
    output_dir: str = MODEL_DIR
    log_dir: str = RESULTS_DIR

    # Logging
    logging_steps: int = 25
    # eval_steps / save_steps are computed dynamically in the training config cell

cfg = Config()
print(f"Config (Google-aligned):")
print(f"  LoRA: r={cfg.lora_r}, alpha={cfg.lora_alpha}, target={cfg.target_modules}")
print(f"  modules_to_save: {cfg.modules_to_save}")
print(f"  Batch: {cfg.batch_size}×{cfg.grad_accum} = {cfg.batch_size*cfg.grad_accum} effective")
print(f"  LR: {cfg.lr}, warmup: {cfg.warmup_ratio}, scheduler: {cfg.lr_scheduler_type}")
print(f"  Epochs: {cfg.num_epochs}, max_grad_norm: {cfg.max_grad_norm}")
print(f"  DoRA: {cfg.use_dora}")

Config (Google-aligned):
  LoRA: r=16, alpha=16, target=all-linear
  modules_to_save: ['lm_head', 'embed_tokens']
  Batch: 4×4 = 16 effective
  LR: 0.0002, warmup: 0.03, scheduler: linear
  Epochs: 1, max_grad_norm: 0.3
  DoRA: False


## Pre-Copy Data to Local SSD

In [4]:
import json, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

LOCAL_TRAIN_DIR = "/content/immunopath_local_v2"
os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)

print("Pre-copying training data to local SSD...")

# --- Check if training_v2 exists, fallback to training ---
if not os.path.exists(cfg.train_path):
    print(f"{cfg.train_path} not found, trying training/ directory...")
    TRAINING_DIR_FALLBACK = f"{DATA_DIR}/training"
    cfg.train_path = f"{TRAINING_DIR_FALLBACK}/train.jsonl"
    cfg.val_path = f"{TRAINING_DIR_FALLBACK}/val.jsonl"
    print(f"Using fallback: {cfg.train_path}")

# Copy JSONL files
for split in ["train.jsonl", "val.jsonl"]:
    src = cfg.train_path.replace("train.jsonl", split)
    dst = f"{LOCAL_TRAIN_DIR}/{split}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  Copied {split}")

# Collect all patch files referenced in train + val
DRIVE_PREFIX = "/content/drive/MyDrive/ImmunoPath"
patch_files = set()
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            for p in sample.get("patch_paths", [])[:cfg.max_patches]:
                if p.startswith(DRIVE_PREFIX):
                    patch_files.add(p)

print(f"  Total patch files referenced: {len(patch_files)}")

# Create directories
dst_dirs = set()
for src_file in patch_files:
    dst_dirs.add(os.path.dirname(src_file.replace(DRIVE_PREFIX, LOCAL_TRAIN_DIR)))
for d in dst_dirs:
    os.makedirs(d, exist_ok=True)

# Copy only missing files
to_copy = []
for src_file in patch_files:
    dst_file = src_file.replace(DRIVE_PREFIX, LOCAL_TRAIN_DIR)
    if not os.path.exists(dst_file):
        to_copy.append((src_file, dst_file))

print(f"  Need to copy: {len(to_copy)} (already cached: {len(patch_files) - len(to_copy)})")

def _copy_one(pair):
    shutil.copy2(pair[0], pair[1])

if to_copy:
    with ThreadPoolExecutor(max_workers=16) as ex:
        list(tqdm(ex.map(_copy_one, to_copy), total=len(to_copy), desc="Copying patches"))

# Rewrite paths in JSONL to point to local SSD
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    updated = []
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            sample["patch_paths"] = [
                p.replace(DRIVE_PREFIX, LOCAL_TRAIN_DIR) if p.startswith(DRIVE_PREFIX) else p
                for p in sample["patch_paths"]
            ]
            updated.append(json.dumps(sample))
    with open(local_path, "w") as f:
        f.write("\n".join(updated) + "\n")

# Validate ALL patch files exist
missing = []
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            for p in sample["patch_paths"][:cfg.max_patches]:
                if not os.path.exists(p):
                    missing.append(p)
if missing:
    print(f"{len(missing)} patches missing! Example: {missing[0]}")
    print("   (Will use available patches — not fatal)")
else:
    print("  All patches verified ✓")

cfg.train_path = f"{LOCAL_TRAIN_DIR}/train.jsonl"
cfg.val_path = f"{LOCAL_TRAIN_DIR}/val.jsonl"
print(f"\nSSD pre-copy complete. Training from {LOCAL_TRAIN_DIR}")

Pre-copying training data to local SSD...
  Copied train.jsonl
  Copied val.jsonl
  Total patch files referenced: 4863
  Need to copy: 4863 (already cached: 0)


Copying patches:   0%|          | 0/4863 [00:00<?, ?it/s]

  All patches verified ✓

SSD pre-copy complete. Training from /content/immunopath_local_v2


## Pre-Cache ALL Images to RAM (A100 Optimization)

**Why:** Each training step currently calls `Image.open()` for every patch in the batch. By pre-loading ALL images into RAM once, we eliminate disk I/O during training.
- ~6K patches × ~1MB = ~6GB RAM, well within Colab Pro's 50GB+
- Workers read the cache via copy-on-write (Linux fork), no memory duplication
- **Expected speedup: 30-50%** (disk I/O was the bottleneck)

In [5]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import time

# Global image cache — workers can read this via copy-on-write after fork()
IMAGE_CACHE = {}

def _load_image(path):
    try:
        img = Image.open(path).convert("RGB")
        img.load()  # Force read into memory (close file handle)
        return path, img
    except Exception:
        return path, None

# Collect ALL unique patch paths from train + val
all_patch_paths = set()
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            for p in sample.get("patch_paths", [])[:cfg.max_patches]:
                all_patch_paths.add(p)

print(f"Pre-caching {len(all_patch_paths)} unique patches to RAM...")
cache_start = time.time()

# Parallel image loading (16 threads for I/O bound work)
with ThreadPoolExecutor(max_workers=16) as pool:
    results = list(tqdm(
        pool.map(_load_image, all_patch_paths),
        total=len(all_patch_paths),
        desc="Caching images"
    ))

for path, img in results:
    if img is not None:
        IMAGE_CACHE[path] = img

cache_elapsed = time.time() - cache_start
cache_mb = sum(img.size[0] * img.size[1] * 3 for img in IMAGE_CACHE.values()) / 1e6
print(f"\n  Cached {len(IMAGE_CACHE)}/{len(all_patch_paths)} images in {cache_elapsed:.1f}s")
print(f"  RAM usage: ~{cache_mb:.0f} MB")
print(f"  ✓ No disk I/O during training — collator uses IMAGE_CACHE")

Pre-caching 4863 unique patches to RAM...


Caching images:   0%|          | 0/4863 [00:00<?, ?it/s]


  Cached 4863/4863 images in 3.4s
  RAM usage: ~3824 MB
  ✓ No disk I/O during training — collator uses IMAGE_CACHE


## Load Model (Google's Exact Configuration)

In [6]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

print(f"Loading {cfg.model_id}...")

# Quantization — matches Google's config exactly
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,  # Google's setting
)

attn_impl = "flash_attention_2" if FLASH_ATTN_AVAILABLE else "eager"

model = AutoModelForImageTextToText.from_pretrained(
    cfg.model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation=attn_impl,
    quantization_config=bnb_config,
)

processor = AutoProcessor.from_pretrained(cfg.model_id)

# FIX: Use right padding for training (Google's approach)
processor.tokenizer.padding_side = "right"

allocated = torch.cuda.memory_allocated() / 1e9
print(f"Model loaded ({attn_impl}), VRAM: {allocated:.2f} GB")
print(f"Tokenizer padding_side: {processor.tokenizer.padding_side}")

Loading google/medgemma-1.5-4b-it...


config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Model loaded (flash_attention_2), VRAM: 3.23 GB
Tokenizer padding_side: right


## LoRA Config

In [8]:
from peft import LoraConfig


peft_config = LoraConfig(
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    r=cfg.lora_r,
    bias="none",
    target_modules=cfg.target_modules,          # "all-linear" — every linear layer
    task_type="CAUSAL_LM",
    modules_to_save=cfg.modules_to_save,         # ["lm_head", "embed_tokens"] — TRAINABLE
)

print(f"LoRA Config:")
print(f"  r={peft_config.r}, alpha={peft_config.lora_alpha}")
print(f"  target_modules: {peft_config.target_modules}")
print(f"  modules_to_save: {peft_config.modules_to_save}")
print(f"  use_dora: {cfg.use_dora}")
print(f"\n  lm_head + embed_tokens will be FULLY TRAINED")


LoRA Config:
  r=16, alpha=16
  target_modules: all-linear
  modules_to_save: ['lm_head', 'embed_tokens']
  use_dora: False

  lm_head + embed_tokens will be FULLY TRAINED


## Build Dataset (HuggingFace Dataset format for SFTTrainer)

In [9]:
import json
from PIL import Image
from datasets import Dataset
from typing import Any

def load_jsonl_to_dataset(path: str, max_patches: int = 4) -> Dataset:
    """Load JSONL and convert to HuggingFace Dataset with 'messages' and 'image' columns."""
    records = []
    with open(path) as f:
        for line in f:
            sample = json.loads(line.strip())
            records.append(sample)
    print(f"Loaded {len(records)} records from {path}")
    return Dataset.from_list(records)


def format_for_sft(example: dict[str, Any]) -> dict[str, Any]:
    """Convert JSONL record to messages format for SFTTrainer collation.

    NOTE: We do NOT load images here — images are loaded lazily in the collator
    from patch_paths. This avoids serializing thousands of PIL images into the
    Arrow cache (slow, memory-heavy, fragile).
    """
    # Count available patches (don't load yet — collator handles that)
    n_images = min(len(example.get("patch_paths", [])), cfg.max_patches)
    if n_images == 0:
        n_images = 1  # collator will create a blank fallback

    # Build messages in the format expected by apply_chat_template
    user_content = [{"type": "image"} for _ in range(n_images)]
    user_content.append({"type": "text", "text": example["prompt"]})

    example["messages"] = [
        {
            "role": "user",
            "content": user_content,
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": example["response"]}],
        },
    ]

    return example


print("Loading datasets...")
train_data = load_jsonl_to_dataset(cfg.train_path, cfg.max_patches)
val_data = load_jsonl_to_dataset(cfg.val_path, cfg.max_patches)

print("Formatting for SFT...")
train_data = train_data.map(format_for_sft, desc="Formatting train")
val_data = val_data.map(format_for_sft, desc="Formatting val")

print(f"\nTrain: {len(train_data)} samples")
print(f"Val:   {len(val_data)} samples")

# Show sample
if len(train_data) > 0:
    s = train_data[0]
    print(f"\nSample messages[0] role: {s['messages'][0]['role']}")
    print(f"Sample messages[1] role: {s['messages'][1]['role']}")
    n_img = sum(1 for c in s['messages'][0]['content'] if c.get('type') == 'image')
    print(f"Sample n_image_placeholders: {n_img}")
    print(f"Sample n_patch_paths: {len(s.get('patch_paths', []))}")

Loading datasets...
Loaded 1502 records from /content/immunopath_local_v2/train.jsonl
Loaded 94 records from /content/immunopath_local_v2/val.jsonl
Formatting for SFT...


Formatting train:   0%|          | 0/1502 [00:00<?, ? examples/s]

Formatting val:   0%|          | 0/94 [00:00<?, ? examples/s]


Train: 1502 samples
Val:   94 samples

Sample messages[0] role: user
Sample messages[1] role: assistant
Sample n_image_placeholders: 4
Sample n_patch_paths: 8


## Custom Collator (Google's Exact Approach)

**KEY DIFFERENCE:**
- v1 masked prompt tokens with `-100` → model never learned what to condition on
- v2 (Google's approach): mask ONLY `pad_token_id`, `boi_token`, and image placeholder (262144)
- The entire conversation (prompt + response) participates in the loss

In [10]:
from typing import Any

def collate_fn(examples: list[dict[str, Any]]):
    """
    Google's exact collation approach (OPTIMIZED with image cache):
    1. apply_chat_template(tokenize=False) → text string
    2. processor(text=..., images=...) → joint tokenization
    3. Mask only pad + image tokens in labels (NOT prompt tokens)

    Uses IMAGE_CACHE for zero-I/O image loading.
    """
    texts = []
    all_images = []

    for example in examples:
        # Use cached images (zero I/O) with fallback to disk
        images = []
        for p in example.get("patch_paths", [])[:cfg.max_patches]:
            if p in IMAGE_CACHE:
                images.append(IMAGE_CACHE[p])
            else:
                try:
                    images.append(Image.open(p).convert("RGB"))
                except Exception:
                    pass
        if not images:
            images = [Image.new("RGB", (512, 512), "white")]

        all_images.append(images)

        # apply_chat_template with tokenize=FALSE → get text string
        text = processor.apply_chat_template(
            example["messages"],
            add_generation_prompt=False,
            tokenize=False,
        ).strip()
        texts.append(text)

    # Joint tokenization: text + images together
    batch = processor(
        text=texts,
        images=all_images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=cfg.max_length,
    )

    # Labels = input_ids, with pad + image tokens masked
    labels = batch["input_ids"].clone()

    # Mask pad tokens
    if processor.tokenizer.pad_token_id is not None:
        labels[labels == processor.tokenizer.pad_token_id] = -100

    # Mask image tokens (boi_token + image placeholder 262144)
    image_token_id = processor.tokenizer.convert_tokens_to_ids(
        processor.tokenizer.special_tokens_map.get("boi_token", "<image_soft_token>")
    )
    if image_token_id is not None:
        labels[labels == image_token_id] = -100
    labels[labels == 262144] = -100

    batch["labels"] = labels

    return batch

print("Collator ready (Google's approach + IMAGE_CACHE optimization)")
print("  ✓ Zero disk I/O — images served from RAM cache")
print("  ✓ Prompt tokens participate in loss")
print("  ✓ Joint tokenization (text + images processed together)")

Collator ready (Google's approach + IMAGE_CACHE optimization)
  ✓ Zero disk I/O — images served from RAM cache
  ✓ Prompt tokens participate in loss
  ✓ Joint tokenization (text + images processed together)


## Training Configuration + SFTTrainer

In [11]:
from trl import SFTConfig, SFTTrainer
import time

# Clean previous local checkpoints
if os.path.exists(LOCAL_CKPT_DIR):
    existing = [d for d in os.listdir(LOCAL_CKPT_DIR) if d.startswith("checkpoint-")]
    if existing:
        print(f"Cleaning {len(existing)} old checkpoints...")
        for d in existing:
            shutil.rmtree(os.path.join(LOCAL_CKPT_DIR, d), ignore_errors=True)

torch.cuda.empty_cache()
gc.collect()

# Estimate total steps for smart eval/save scheduling
est_steps = len(train_data) // (cfg.batch_size * cfg.grad_accum)
eval_every = max(25, est_steps // 3)  # ~3 evals per epoch
print(f"Estimated steps: ~{est_steps}, eval every {eval_every} steps")

# --- SFTConfig (Google's exact training args + A100 optimizations) ---
training_args = SFTConfig(
    # OUTPUT: Local SSD (not GDrive!) — 10-50x faster writes
    output_dir=LOCAL_CKPT_DIR,

    # Training schedule
    num_train_epochs=cfg.num_epochs,                       # 1 epoch
    per_device_train_batch_size=cfg.batch_size,             # 4
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=cfg.grad_accum,             # 4 → effective 16

    # Optimizer — Google's exact settings
    learning_rate=cfg.lr,                                   # 2e-4
    warmup_ratio=cfg.warmup_ratio,                          # 0.03
    weight_decay=cfg.weight_decay,
    max_grad_norm=cfg.max_grad_norm,                        # 0.3
    lr_scheduler_type=cfg.lr_scheduler_type,                # "linear"
    optim="adamw_torch_fused",                              # Fused optimizer (A100 optimized)
    bf16=True,

    # Eval/Save — tuned to actual step count
    logging_steps=cfg.logging_steps,
    eval_strategy="steps",
    eval_steps=eval_every,
    save_strategy="steps",
    save_steps=eval_every,                                  # Match eval_steps
    save_total_limit=2,

    # Logging — local SSD (not GDrive!)
    report_to="tensorboard",
    logging_dir=f"{LOCAL_LOG_DIR}/tb_logs",

    # Best model tracking
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Memory optimizations
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Google's setting

    # Dataset handling
    dataset_kwargs={"skip_prepare_dataset": True},           # Google's setting
    remove_unused_columns=False,                             # Keep columns for collator
    label_names=["labels"],                                  # Google's setting

    # A100 DATA LOADING OPTIMIZATIONS
    dataloader_num_workers=4,                                # 4 workers (up from 2)
    dataloader_pin_memory=True,                              # Pin memory for faster GPU transfer
    dataloader_prefetch_factor=4,                            # Pre-fetch 4 batches per worker
)

# --- SFTTrainer (FIX #5: proper PEFT integration) ---
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn,
)

# Print trainable parameters
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in trainer.model.parameters())
print(f"\nSFTTrainer ready:")
print(f"  Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"  Effective batch: {cfg.batch_size * cfg.grad_accum}")
print(f"  Train samples: {len(train_data)}")
print(f"  Val samples: {len(val_data)}")
print(f"  Estimated steps: ~{est_steps}")
print(f"  Eval/save every {eval_every} steps")

allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
print(f"  VRAM: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")

print(f"\n  ⚡ A100 OPTIMIZATIONS:")
print(f"  ✓ Checkpoints → local SSD ({LOCAL_CKPT_DIR})")
print(f"  ✓ TensorBoard → local SSD ({LOCAL_LOG_DIR})")
print(f"  ✓ Image cache: {len(IMAGE_CACHE)} patches in RAM (zero I/O)")
print(f"  ✓ dataloader_num_workers=4, prefetch_factor=4, pin_memory=True")
print(f"  ✓ TF32 + Flash Attention 2 + bf16")
print(f"  ✓ adamw_torch_fused optimizer")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Estimated steps: ~93, eval every 31 steps


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)



SFTTrainer ready:
  Trainable params: 1,381,002,752 / 2,966,297,456 (46.56%)
  Effective batch: 16
  Train samples: 1502
  Val samples: 94
  Estimated steps: ~93
  Eval/save every 31 steps
  VRAM: 5.99 GB allocated, 6.47 GB reserved

  ⚡ A100 OPTIMIZATIONS:
  ✓ Checkpoints → local SSD (/content/immunopath_ckpts_v2)
  ✓ TensorBoard → local SSD (/content/immunopath_logs_v2)
  ✓ Image cache: 4863 patches in RAM (zero I/O)
  ✓ dataloader_num_workers=4, prefetch_factor=4, pin_memory=True
  ✓ TF32 + Flash Attention 2 + bf16
  ✓ adamw_torch_fused optimizer


## TRAIN

In [12]:
print("=" * 60)
print("STARTING FINE-TUNING (v2 — Google-aligned)")
print("=" * 60)
print(f"  Model:     {cfg.model_id}")
print(f"  LoRA:      r={cfg.lora_r}, alpha={cfg.lora_alpha}")
print(f"  modules_to_save: {cfg.modules_to_save}")
print(f"  target:    {cfg.target_modules}")
print(f"  Batch:     {cfg.batch_size}×{cfg.grad_accum}={cfg.batch_size*cfg.grad_accum}")
print(f"  Epochs:    {cfg.num_epochs}")
print(f"  LR:        {cfg.lr}")
print(f"  Grad norm: {cfg.max_grad_norm}")
print("=" * 60)

start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

print(f"\n{'=' * 60}")
print(f"TRAINING COMPLETE in {elapsed/60:.1f} minutes")
print(f"Final train loss: {train_result.training_loss:.4f}")
print(f"{'=' * 60}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


STARTING FINE-TUNING (v2 — Google-aligned)
  Model:     google/medgemma-1.5-4b-it
  LoRA:      r=16, alpha=16
  modules_to_save: ['lm_head', 'embed_tokens']
  target:    all-linear
  Batch:     4×4=16
  Epochs:    1
  LR:        0.0002
  Grad norm: 0.3


Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss
31,0.550155,0.067015
62,0.065929,0.064609
93,0.062119,0.061820



TRAINING COMPLETE in 38.4 minutes
Final train loss: 0.1928


## Save LoRA Adapters

In [13]:
# Save adapters to LOCAL SSD first (fast), then copy to GDrive
local_adapter_dir = f"{LOCAL_CKPT_DIR}/lora_adapters"
os.makedirs(local_adapter_dir, exist_ok=True)

# Save locally (instant)
trainer.save_model(local_adapter_dir)
processor.save_pretrained(local_adapter_dir)
print(f"Adapters saved locally: {local_adapter_dir}")

# Also save full checkpoint locally
trainer.save_model(LOCAL_CKPT_DIR)
processor.save_pretrained(LOCAL_CKPT_DIR)

# Now copy to GDrive (slower but persistent)
print("\nCopying adapters to GDrive...")
gdrive_adapter_dir = f"{cfg.output_dir}/lora_adapters"
os.makedirs(gdrive_adapter_dir, exist_ok=True)

import shutil
# Copy adapter files
for f_name in os.listdir(local_adapter_dir):
    src = os.path.join(local_adapter_dir, f_name)
    dst = os.path.join(gdrive_adapter_dir, f_name)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
print(f"  → {gdrive_adapter_dir}")

# Also copy to model root on GDrive
for f_name in os.listdir(LOCAL_CKPT_DIR):
    src = os.path.join(LOCAL_CKPT_DIR, f_name)
    dst = os.path.join(cfg.output_dir, f_name)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
print(f"  → {cfg.output_dir}")

saved_files = os.listdir(gdrive_adapter_dir)
print(f"\nAdapter files: {saved_files}")
print("✓ Adapters saved to both local SSD and GDrive")

Adapters saved locally: /content/immunopath_ckpts_v2/lora_adapters

Copying adapters to GDrive...
  → /content/drive/MyDrive/ImmunoPath/models/immunopath-v2/lora_adapters
  → /content/drive/MyDrive/ImmunoPath/models/immunopath-v2

Adapter files: ['tokenizer_config.json', 'processor_config.json', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'training_args.bin', 'tokenizer.json']
✓ Adapters saved to both local SSD and GDrive


## Save Training Log

In [14]:
from datetime import datetime

log_history = trainer.state.log_history
train_logs = [l for l in log_history if "loss" in l and "eval_loss" not in l]
eval_logs = [l for l in log_history if "eval_loss" in l]

training_report = {
    "phase": "5_v2",
    "timestamp": datetime.now().isoformat(),
    "model_id": cfg.model_id,
    "peft_method": "LoRA (via SFTTrainer + modules_to_save)",
    "critical_fixes": [
        "modules_to_save=['lm_head', 'embed_tokens'] — output head is trainable",
        "NO prompt masking — only pad+image tokens masked (Google approach)",
        "1 epoch (not 3) — prevents structure overfitting",
        "max_grad_norm=0.3 (not 1.0) — QLoRA paper",
        "SFTTrainer (not raw Trainer) — proper PEFT integration",
        "target_modules='all-linear' — full linear coverage",
        "Effective batch 16 (not 8) — stable gradients",
    ],
    "a100_optimizations": [
        "Image pre-caching to RAM (zero I/O during training)",
        "Checkpoints on local SSD (not GDrive)",
        "TF32 + Flash Attention 2 + bf16",
        "dataloader_num_workers=4, prefetch_factor=4, pin_memory=True",
        "adamw_torch_fused optimizer",
        "TOKENIZERS_PARALLELISM=false",
    ],
    "peft_config": {
        "r": cfg.lora_r,
        "alpha": cfg.lora_alpha,
        "dropout": cfg.lora_dropout,
        "target_modules": cfg.target_modules,
        "modules_to_save": cfg.modules_to_save,
        "use_dora": cfg.use_dora,
    },
    "training_config": {
        "batch_size": cfg.batch_size,
        "grad_accum": cfg.grad_accum,
        "effective_batch": cfg.batch_size * cfg.grad_accum,
        "lr": cfg.lr,
        "epochs": cfg.num_epochs,
        "warmup_ratio": cfg.warmup_ratio,
        "max_grad_norm": cfg.max_grad_norm,
        "lr_scheduler": cfg.lr_scheduler_type,
        "optimizer": "adamw_torch_fused",
        "attn_implementation": attn_impl,
    },
    "data": {
        "train_samples": len(train_data),
        "val_samples": len(val_data),
        "image_cache_size": len(IMAGE_CACHE),
    },
    "results": {
        "final_train_loss": train_result.training_loss,
        "best_eval_loss": min((l["eval_loss"] for l in eval_logs), default=None),
        "training_time_minutes": round(elapsed / 60, 1),
    },
    "train_logs": train_logs,
    "eval_logs": eval_logs,
}

# Save locally (fast) then to GDrive (persistent)
local_log_path = f"{LOCAL_LOG_DIR}/training_log.json"
with open(local_log_path, 'w') as f:
    json.dump(training_report, f, indent=2, default=str)

gdrive_log_path = f"{RESULTS_DIR}/training_log.json"
shutil.copy2(local_log_path, gdrive_log_path)
print(f"Training log saved to {gdrive_log_path}")

# Summary
best_eval = training_report["results"]["best_eval_loss"]
print(f"\n  Final train loss:  {train_result.training_loss:.4f}")
print(f"  Best eval loss:    {best_eval}")
print(f"  Training time:     {elapsed/60:.1f} min")
print(f"  Images cached:     {len(IMAGE_CACHE)}")

Training log saved to /content/drive/MyDrive/ImmunoPath/results/phase5_v2/training_log.json

  Final train loss:  0.1928
  Best eval loss:    0.06181960552930832
  Training time:     38.4 min
  Images cached:     4863


## Quick Inference Test

In [15]:
# Switch padding to LEFT for inference
processor.tokenizer.padding_side = "left"

# Load a val sample for testing
val_jsonl = f"{LOCAL_TRAIN_DIR}/val.jsonl"
val_record = None
if os.path.exists(val_jsonl):
    with open(val_jsonl) as f:
        val_record = json.loads(f.readline().strip())

if val_record:
    images = []
    for p in val_record["patch_paths"][:cfg.max_patches]:
        try:
            images.append(Image.open(p).convert("RGB"))
        except Exception:
            pass
    if not images:
        images = [Image.new("RGB", (512, 512), "white")]

    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": val_record["prompt"]})
    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    device = next(p.device for p in trainer.model.parameters() if p.device.type != 'meta')
    inputs = {
        k: v.to(device, dtype=torch.bfloat16) if v.is_floating_point()
        else v.to(device)
        for k, v in inputs.items()
        if torch.is_tensor(v)
    }
    input_len = inputs["input_ids"].shape[-1]

    trainer.model.eval()
    with torch.inference_mode():
        output_ids = trainer.model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            use_cache=True,
        )

    generated = output_ids[0][input_len:]
    response_text = processor.decode(generated, skip_special_tokens=True).strip()

    for img in images:
        img.close()

    print("INFERENCE TEST:")
    print(f"  Output: {response_text[:400]}")
    print()

    # Parse JSON
    try:
        clean = response_text
        if "```json" in clean:
            clean = clean.split("```json")[1].split("```")[0]
        elif "```" in clean:
            clean = clean.split("```")[1].split("```")[0]
        start = clean.find("{")
        end = clean.rfind("}") + 1
        if start != -1 and end > start:
            parsed = json.loads(clean[start:end])
            print(f"  Valid JSON: {list(parsed.keys())}")
            print(f"  Parsed: {json.dumps(parsed, indent=2)}")
        else:
            print("  No JSON found in output")
    except json.JSONDecodeError as e:
        print(f"  JSON parse error: {e}")

    # Ground truth
    print(f"\n  Ground truth:")
    gt = json.loads(val_record["response"])
    print(f"  {json.dumps(gt, indent=2)}")
else:
    print("No val samples available")

# Reset padding to right (in case of further training)
processor.tokenizer.padding_side = "right"

INFERENCE TEST:
  Output: {"cd274_expression": "high", "msi_status": "MSS", "tme_subtype": "IE", "immune_phenotype": "inflamed", "cd8_infiltration": "high", "til_density": "high", "til_fraction": 0.72, "immune_score": 0.766}

  Valid JSON: ['cd274_expression', 'msi_status', 'tme_subtype', 'immune_phenotype', 'cd8_infiltration', 'til_density', 'til_fraction', 'immune_score']
  Parsed: {
  "cd274_expression": "high",
  "msi_status": "MSS",
  "tme_subtype": "IE",
  "immune_phenotype": "inflamed",
  "cd8_infiltration": "high",
  "til_density": "high",
  "til_fraction": 0.72,
  "immune_score": 0.766
}

  Ground truth:
  {
  "cd274_expression": "high",
  "msi_status": "MSS",
  "tme_subtype": "F",
  "immune_phenotype": "desert",
  "cd8_infiltration": "low",
  "til_density": "moderate",
  "til_fraction": 0.442,
  "immune_score": 0.322
}


## Phase 5 v2 Summary

In [16]:
print("\n" + "=" * 60)
print("PHASE 5 v2 — FINE-TUNING COMPLETE")
print("=" * 60)
print(f"\n  Model:       {cfg.model_id}")
print(f"  PEFT:        LoRA (r={cfg.lora_r}, α={cfg.lora_alpha})")
print(f"  CRITICAL:    modules_to_save={cfg.modules_to_save}")
print(f"  CRITICAL:    target_modules={cfg.target_modules}")
print(f"  Train:       {len(train_data)} samples × {cfg.num_epochs} epoch")
print(f"  Val:         {len(val_data)} samples")
print(f"  Final loss:  {train_result.training_loss:.4f}")
print(f"  Best eval:   {best_eval}")
print(f"  Time:        {elapsed/60:.1f} min")
print(f"  Adapters:    {gdrive_adapter_dir}")
print(f"\n{'=' * 60}")
print("NEXT: Phase 6 v2 — Evaluation")
print(f"{'=' * 60}")


PHASE 5 v2 — FINE-TUNING COMPLETE

  Model:       google/medgemma-1.5-4b-it
  PEFT:        LoRA (r=16, α=16)
  CRITICAL:    modules_to_save=['lm_head', 'embed_tokens']
  CRITICAL:    target_modules=all-linear
  Train:       1502 samples × 1 epoch
  Val:         94 samples
  Final loss:  0.1928
  Best eval:   0.06181960552930832
  Time:        38.4 min
  Adapters:    /content/drive/MyDrive/ImmunoPath/models/immunopath-v2/lora_adapters

NEXT: Phase 6 v2 — Evaluation
